# Prediction of Atrial Fibrillation Termination Using Dominant Frequency in the Maze Procedure

Analysis code accompanying the manuscript (see README.md)

All inference uses **patient-level cluster bootstrap**. Conventional methods that assume independent
observations (Mann-Whitney U test, DeLong test) are shown for reference only.

## Contents
1. Setup
2. DF calculation (QRST cancellation → FFT)
3. Quality of QRST cancellation
4. Study population and group comparison
5. Primary analysis (prediction models)
6. Model comparison
7. Sensitivity analyses
8. Sample size estimation
9. Post hoc exploratory analyses
10. Saving results and software versions

## Environment
Python 3.10, NumPy, pandas, SciPy, scikit-learn, matplotlib, tqdm, openpyxl (for reading the patient table).
ECG-FM embeddings are loaded from files extracted in a separate notebook (01_ECG-FM_extraction.ipynb).
Order of execution: (1) 01_ECG-FM_extraction.ipynb → (2) this notebook

## 1. Setup

In [ ]:
import os, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
from scipy.stats import mannwhitneyu, spearmanr, pearsonr, fisher_exact, kruskal, norm
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# ---- Analysis settings (prespecified) ----
ROOT        = "path/to/project"   # <- change to your environment
LEAD        = 'Ⅱ'              # analysis lead (based on Uetake et al.)
FS          = 500              # sampling frequency [Hz]
AF_BAND     = (3.5, 9.0)       # DF search band [Hz]
NFFT        = 2**16            # FFT length (common to all recording lengths)
WINDOW      = np.hanning       # window function
UV_PER_COUNT = 4.88            # voltage per count [μV]

N_SPLITS    = 5                # number of CV folds
N_REPEATS   = 10               # number of CV repetitions
N_BOOT      = 2000             # number of bootstrap resamples
SEED        = 42
C_PARAM     = 1.0              # L2 regularization parameter
N_PCA       = 3                # number of principal components of ECG-FM embedding

LEADS_12 = ['Ⅰ','Ⅱ','Ⅲ','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']

print("Analysis settings loaded")

In [ ]:
# Load metadata
metadata = pd.read_csv(os.path.join(ROOT, "data/metadata.csv"), encoding="utf-8-sig")
metadata.columns = metadata.columns.str.strip()
metadata = metadata.dropna(subset=['ecg_path', 'ID', 'Result']).reset_index(drop=True)

labels      = metadata['Result'].values.astype(int)   # 1 = termination, 0 = non-termination
patient_ids = metadata['ID'].values

print("Columns:", metadata.columns.tolist())
print(f"{len(metadata)} ECGs / {len(np.unique(patient_ids))} patients")
print("Labels:", dict(zip(*np.unique(labels, return_counts=True))))

# Remove ECG files with identical content
import hashlib
def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

_hash = metadata['ecg_path'].map(file_hash)
KEEP = ~_hash.duplicated(keep='first')          # drop the second and later copies
print(f"Removed as duplicates: {(~KEEP).sum()} ECG(s) -> {metadata.loc[~KEEP, 'ID'].tolist()}")

metadata    = metadata[KEEP].reset_index(drop=True)
labels      = metadata['Result'].values.astype(int)
patient_ids = metadata['ID'].values
print(f"{len(metadata)} ECGs / {len(np.unique(patient_ids))} patients")

## 2. DF calculation

QRST cancellation is based on the template matching and subtraction method of Bollmann et al.,
incorporating the findings of Lemay et al. (morphology-specific templates and optimization of amplitude and timing).

In [ ]:
def detect_r_peaks(sig, fs=FS, rescue=True):
    '''R-peak detection based on the Pan-Tompkins algorithm (with recovery of missed beats)'''
    b, a = butter(2, [5, 15], btype='band', fs=fs)
    filtered = filtfilt(b, a, sig)
    squared = np.diff(filtered) ** 2
    win = int(0.15 * fs)
    integrated = np.convolve(squared, np.ones(win)/win, mode='same')
    thr = np.mean(integrated) + 0.5 * np.std(integrated)
    peaks, _ = find_peaks(integrated, height=thr, distance=int(0.25*fs))

    # Re-search with half the threshold where the RR interval exceeds 1.6 x the median
    if rescue and len(peaks) >= 3:
        med_rr = np.median(np.diff(peaks))
        extra = []
        for p0, p1 in zip(peaks[:-1], peaks[1:]):
            if (p1 - p0) > 1.6 * med_rr:
                lo, hi = p0 + int(0.25*fs), p1 - int(0.25*fs)
                if hi > lo:
                    cand, _ = find_peaks(integrated[lo:hi], height=thr*0.5,
                                         distance=int(0.25*fs))
                    extra += [lo + c for c in cand]
        if extra:
            peaks = np.array(sorted(set(list(peaks) + extra)))

    # Refine each peak to the R-wave apex on the filtered signal
    adjust = int(0.03 * fs)
    r_peaks = []
    for p in peaks:
        lo, hi = max(0, p-adjust), min(len(sig), p+adjust)
        if hi > lo:
            r_peaks.append(lo + np.argmax(np.abs(sig[lo:hi])))
    return np.array(sorted(set(r_peaks)))

In [ ]:
def qrst_subtract(sig, r_peaks, fs=FS, min_cluster=4, return_internals=False):
    '''QRST cancellation with multiple templates

    - The cancellation window is set from the median RR interval and truncated before the next beat
    - Beats are clustered by QRS morphology, with at least 4 beats per cluster
    - Template timing (±12 ms) and amplitude (least squares) are optimized for each beat
    '''
    if len(r_peaks) < 5:
        r = (None, 0, 'too_few_beats')
        return r + ({},) if return_internals else r

    rr = np.diff(r_peaks)
    median_rr = int(np.median(rr))
    pre  = int(0.10 * fs)
    post = min(int(0.45*fs), max(int(0.15*fs), median_rr - pre - 1))

    # --- Pass 1: template construction (beats with a full-length window only) ---
    segs = []
    for i, r in enumerate(r_peaks):
        if r - pre < 0 or r + post >= len(sig):
            continue
        if i + 1 < len(r_peaks) and (r_peaks[i+1] - r) < (pre + post):
            continue
        segs.append(sig[r-pre : r+post])

    if len(segs) < 5:
        segs = [sig[r-pre : r+post] for r in r_peaks
                if r-pre >= 0 and r+post < len(sig)]
        if len(segs) < 3:
            r = (None, 0, 'too_few_segments')
            return r + ({},) if return_internals else r
    segments = np.array(segs)
    T = segments.shape[1]

    # Features: amplitude-normalized QRS segment (40 ms before to 60 ms after the R peak)
    qrs_lo, qrs_hi = pre - int(0.04*fs), pre + int(0.06*fs)
    def qrs_feat(x):
        v = x[max(0, qrs_lo):qrs_hi]
        return v / (np.linalg.norm(v) + 1e-9)

    feats = np.array([qrs_feat(s) for s in segments])
    n_clusters = max(1, min(3, len(segments) // min_cluster))
    if n_clusters >= 2:
        cl = KMeans(n_clusters=n_clusters, random_state=SEED,
                    n_init=10).fit_predict(feats)
        for c in np.unique(cl):                      # merge small clusters into the overall mean
            if np.sum(cl == c) < min_cluster:
                cl[cl == c] = -1
    else:
        cl = np.zeros(len(segments), dtype=int)

    templates = {c: (segments.mean(axis=0) if c == -1 else segments[cl == c].mean(axis=0))
                 for c in np.unique(cl)}
    keys = list(templates.keys())
    tmpl_feats = {c: qrs_feat(templates[c]) for c in keys}

    # --- Pass 2: subtraction from all beats ---
    residual = sig.copy()
    used = []
    max_shift = int(0.012 * fs)

    for i, r in enumerate(r_peaks):
        start = max(0, r - pre)
        end   = min(len(sig), r + post)
        if i + 1 < len(r_peaks):
            end = min(end, max(start + 1, r_peaks[i+1] - pre))
        if end - start < int(0.06 * fs):
            continue

        offset = start - (r - pre)
        L = end - start
        if offset < 0 or offset + L > T:
            L = min(L, T - offset)
            if L <= 0:
                continue
            end = start + L

        seg_full = np.zeros(T)
        seg_full[offset:offset+L] = sig[start:end]
        f = qrs_feat(seg_full)
        best = keys[int(np.argmax([float(np.dot(f, tmpl_feats[c])) for c in keys]))]
        seg = sig[start:end]

        # Fine alignment in time
        best_shift, best_corr = 0, -np.inf
        for sh in range(-max_shift, max_shift + 1):
            lo = offset + sh
            if lo < 0 or lo + L > T:
                continue
            c = float(np.dot(seg, templates[best][lo:lo+L]))
            if c > best_corr:
                best_corr, best_shift = c, sh

        t = templates[best][offset+best_shift : offset+best_shift+L]

        # Amplitude scaling (least squares, limited to 0.5-1.5)
        denom = float(np.dot(t, t))
        alpha = float(np.dot(seg, t)) / denom if denom > 1e-12 else 1.0
        alpha = np.clip(alpha, 0.5, 1.5)

        residual[start:end] -= alpha * t
        used.append(r)

    if return_internals:
        return residual, len(used), 'ok', dict(
            pre=pre, post=post, segments=segments, clusters=cl,
            templates=templates, valid_peaks=used)
    return residual, len(used), 'ok'

In [ ]:
def preprocess(csv_path, lead=LEAD, fs=FS):
    '''Read CSV -> band-pass filter (0.5-40 Hz, zero-phase)'''
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    if lead not in df.columns:
        return None
    sig = df[lead].values.astype(np.float64)
    b, a = butter(4, [0.5, 40], btype='band', fs=fs)
    return filtfilt(b, a, sig)


def spectrum(residual, fs=FS, window=None, smooth_hz=None):
    '''Residual -> window -> FFT -> spectrum in the AF band'''
    wf = window if window is not None else WINDOW
    w = residual * wf(len(residual))
    fv = np.abs(np.fft.rfft(w, n=NFFT))
    fr = np.fft.rfftfreq(NFFT, d=1/fs)
    if smooth_hz:
        win = max(1, int(smooth_hz / (fr[1] - fr[0])))
        fv = np.convolve(fv, np.ones(win)/win, mode='same')
    m = (fr >= AF_BAND[0]) & (fr <= AF_BAND[1])
    return fr, fv, m


def compute_df(csv_path, lead=LEAD, fs=FS, window=None, smooth_hz=None,
               return_all=False):
    '''Calculate DF

    Returns DF [Hz]; with return_all=True, also heart rate, number of beats used, status and organization index
    '''
    sig = preprocess(csv_path, lead, fs)
    if sig is None:
        return (np.nan,)*5 if return_all else np.nan

    r_peaks = detect_r_peaks(sig, fs)
    hr = len(r_peaks) / (len(sig)/fs) * 60
    residual, n_used, status = qrst_subtract(sig, r_peaks, fs)
    if residual is None:
        return (np.nan, hr, n_used, status, np.nan) if return_all else np.nan

    fr, fv, m = spectrum(residual, fs, window, smooth_hz)
    bf, bp = fr[m], fv[m]
    df_value = bf[np.argmax(bp)]
    org = bp.max() / (bp.mean() + 1e-12)      # spectral peakedness

    return (df_value, hr, n_used, status, org) if return_all else df_value

In [ ]:
# Calculate DF for all recordings
rows = [compute_df(p, return_all=True)
        for p in tqdm(metadata['ecg_path'], desc="DF")]

DF_all     = np.array([r[0] for r in rows])
hr_all     = np.array([r[1] for r in rows])
nused_all  = np.array([r[2] for r in rows])
status_all = np.array([r[3] for r in rows])
org_all    = np.array([r[4] for r in rows])

print("\n=== Calculation status ===")
for k, v in collections.Counter(status_all).items():
    print(f"  {k}: {v}")
print(f"Heart rate : median {np.median(hr_all):.0f}, range {hr_all.min():.0f}–{hr_all.max():.0f} /min")
print(f"Beats used : median {np.median(nused_all):.0f}, range {nused_all.min()}–{nused_all.max()}")
print(f"DF         : median {np.nanmedian(DF_all):.2f}, range {np.nanmin(DF_all):.2f}–{np.nanmax(DF_all):.2f} Hz")
print(f"  At the lower band limit (≤3.55 Hz): {np.sum(DF_all <= 3.55)}")
print(f"  Near the upper band limit (≥8.9 Hz): {np.sum(DF_all >= 8.9)}")

## 3. Quality of QRST cancellation

**QRS residue rate** (according to Lemay et al.): a threshold ς = m + 2·IQR/0.6745 (m, median) is derived from the
amplitude distribution of the residual signal outside the QRS intervals (±60 ms around each R peak). Beats whose QRS
interval contains absolute values exceeding ς are classified as having significant QRS residue, and their proportion is the QRS residue rate.

**Suppression ratio** (auxiliary index defined in this study): the median, across beats, of the maximum absolute amplitude
within the cancellation window before cancellation divided by that after cancellation.

Quality indices are used only for description and group comparison, not for exclusion.

In [ ]:
def quality_metrics(csv_path, lead=LEAD, fs=FS, qrs_half_ms=60):
    '''Suppression ratio, QRS residue rate and longest near-zero run'''
    sig = preprocess(csv_path, lead, fs)
    if sig is None:
        return np.nan, np.nan, np.nan
    r_peaks = detect_r_peaks(sig, fs)
    residual, n_used, status, internals = qrst_subtract(
        sig, r_peaks, fs, return_internals=True)
    if residual is None:
        return np.nan, np.nan, np.nan

    pre, post = internals['pre'], internals['post']
    peaks = internals['valid_peaks']
    n = len(residual)

    # Suppression ratio (reduction of the maximum QRST amplitude by cancellation)
    before = [np.abs(sig[max(0,r-pre):min(n,r+post)]).max() for r in peaks]
    after  = [np.abs(residual[max(0,r-pre):min(n,r+post)]).max() for r in peaks]
    supp = np.median(before) / (np.median(after) + 1e-9)

    # QRS residue rate (method of Lemay et al.)
    qh = int(qrs_half_ms/1000*fs)
    bad = np.zeros(n, dtype=bool)
    for r in peaks:
        bad[max(0, r-qh):min(n, r+qh)] = True
    aa = residual[~bad]
    if len(aa) < fs:
        return supp, np.nan, np.nan
    thr = np.median(aa) + 2 * (np.percentile(aa,75)-np.percentile(aa,25))/0.6745
    n_sig = sum(np.max(np.abs(residual[max(0,r-qh):min(n,r+qh)])) > thr
                for r in peaks if min(n,r+qh) > max(0,r-qh))
    rate = n_sig / len(peaks) * 100 if peaks else np.nan

    # Longest near-zero run (detection of cancellation failure)
    near0 = np.abs(residual) < 0.02 * np.std(sig)
    run = best = 0
    for v in near0:
        run = run + 1 if v else 0
        best = max(best, run)
    return supp, rate, best/fs*1000

In [ ]:
qc_rows = [quality_metrics(p) for p in tqdm(metadata['ecg_path'], desc="Quality")]
supp_all = np.array([q[0] for q in qc_rows])
rate_all = np.array([q[1] for q in qc_rows])
zero_all = np.array([q[2] for q in qc_rows])

print("\n=== Quality of QRST cancellation (all recordings) ===")
print(f"QRS residue rate : median {np.nanmedian(rate_all):.1f}%, mean {np.nanmean(rate_all):.1f}%")
print(f"Suppression ratio: median {np.nanmedian(supp_all):.1f}, range {np.nanmin(supp_all):.2f}–{np.nanmax(supp_all):.2f}")
print(f"Recordings with suppression ratio < 2: {np.sum(supp_all < 2)}")

## 4. Study population and group comparison

Exclusion criteria (prespecified): fewer than 5 detected R peaks, or fewer than 3 beats available for template construction.

Note: duplicated ECG files in the mapping table are removed in Section 1. The prespecified exclusion criteria are applied below.

Inference is based on patient-level cluster bootstrap.
The Mann-Whitney U test assumes independent observations and is shown for reference only.

In [ ]:
# Apply exclusion criteria
valid = ~np.isnan(DF_all)

DF_v   = DF_all[valid]
y_v    = labels[valid]
pid_v  = patient_ids[valid]
LAD_v  = metadata['LAD'].values.astype(float)[valid]
Age_v  = metadata['Age'].values.astype(float)[valid]
hr_v   = hr_all[valid]
org_v  = org_all[valid]
supp_v = supp_all[valid]
rate_v = rate_all[valid]

n_excl = (~valid).sum()
print(f"Recordings meeting exclusion criteria: {n_excl}")
print(f"Analyzed: {valid.sum()} ECGs / {len(np.unique(pid_v))} patients")
print("Labels:", dict(zip(*np.unique(y_v, return_counts=True))))
print("\nMissing:", {'DF': int(np.isnan(DF_v).sum()),
                 'LAD': int(np.isnan(LAD_v).sum()),
                 'Age': int(np.isnan(Age_v).sum())})

# Recording length
lengths = np.array([sum(1 for _ in open(p, 'rb')) - 1 for p in metadata['ecg_path']])[valid]
print("\nRecording length:", {f"{int(v/FS)} s": int(c) for v, c in zip(*np.unique(lengths, return_counts=True))})
long_ = lengths >= 7500            # 15 s or longer
tab_len = np.array([[np.sum(long_ & (y_v==1)), np.sum(~long_ & (y_v==1))],
                    [np.sum(long_ & (y_v==0)), np.sum(~long_ & (y_v==0))]])
print(f"Group difference (≥15 s vs 10 s, Fisher exact test): p = {fisher_exact(tab_len)[1]:.3f}")

In [ ]:
def cluster_bootstrap(y, groups, statistic, n_boot=N_BOOT, seed=SEED):
    '''Patient-level cluster bootstrap

    Evaluates statistic(idx) -> scalar n_boot times and returns the observed value,
    95% CI (percentile method) and two-sided p value (null value 0)
    '''
    rng = np.random.default_rng(seed)
    pts = np.unique(groups)
    idx_by_pt = {p: np.where(groups == p)[0] for p in pts}
    obs = statistic(np.arange(len(y)))
    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([idx_by_pt[p]
                              for p in rng.choice(pts, len(pts), replace=True)])
        if len(np.unique(y[idx])) < 2:
            continue
        v = statistic(idx)
        if not np.isnan(v):
            vals.append(v)
    vals = np.array(vals)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    p = min(1.0, 2 * min((vals <= 0).mean(), (vals >= 0).mean()))
    return obs, lo, hi, p, vals


def cluster_bootstrap_auc(y, score, groups, n_boot=N_BOOT, seed=SEED):
    '''Observed AUROC and 95% CI (null value 0.5)'''
    rng = np.random.default_rng(seed)
    pts = np.unique(groups)
    idx_by_pt = {p: np.where(groups == p)[0] for p in pts}
    obs = roc_auc_score(y, score)
    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([idx_by_pt[p]
                              for p in rng.choice(pts, len(pts), replace=True)])
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(roc_auc_score(y[idx], score[idx]))
    vals = np.array(vals)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    p = min(1.0, 2 * min((vals <= 0.5).mean(), (vals >= 0.5).mean()))
    return obs, lo, hi, p

In [ ]:
# Group comparison of QRST cancellation quality (patient-level cluster bootstrap)
# Mean difference for the QRS residue rate (0% in most recordings); median difference for the suppression ratio
print("=== Group comparison of QRST cancellation quality ===\n")

stat_rate = lambda idx: (np.nanmean(rate_v[idx][y_v[idx]==1]) - np.nanmean(rate_v[idx][y_v[idx]==0]))
d, lo, hi, p, _ = cluster_bootstrap(y_v, pid_v, stat_rate)
print(f"QRS residue rate: median termination {np.nanmedian(rate_v[y_v==1]):.1f}% / non-termination {np.nanmedian(rate_v[y_v==0]):.1f}%, "
      f"mean termination {np.nanmean(rate_v[y_v==1]):.1f}% / non-termination {np.nanmean(rate_v[y_v==0]):.1f}%")
print(f"  Mean difference {d:+.1f}% [95% CI {lo:+.1f}, {hi:+.1f}], p≈{p:.3f}")

stat_supp = lambda idx: (np.nanmedian(supp_v[idx][y_v[idx]==1]) - np.nanmedian(supp_v[idx][y_v[idx]==0]))
d, lo, hi, p, _ = cluster_bootstrap(y_v, pid_v, stat_supp)
print(f"Suppression ratio: median termination {np.nanmedian(supp_v[y_v==1]):.1f} / non-termination {np.nanmedian(supp_v[y_v==0]):.1f}")
print(f"  Median difference {d:+.1f} [95% CI {lo:+.1f}, {hi:+.1f}], p≈{p:.3f}")

m = ~np.isnan(rate_v)
print(f"\nQRS residue rate vs DF: Spearman rho = {spearmanr(rate_v[m], DF_v[m])[0]:+.3f}")
print(f"Recordings with QRS residue rate > 20%: {np.sum(rate_v > 20)}")

In [ ]:
# Distribution of the QRS residue rate (is the mean driven by a few recordings?)
r = rate_v[~np.isnan(rate_v)]
q1, q2, q3 = np.percentile(r, [25, 50, 75])
print(f"QRS residue rate (n={len(r)}): median {q2:.1f}% (IQR {q1:.1f}–{q3:.1f}), "
      f"range {r.min():.1f}–{r.max():.1f}%, mean {r.mean():.1f}%")
print(f"  0%: {np.sum(r == 0)} ECGs ({np.mean(r == 0)*100:.0f}%)")
for t in (10, 20, 30, 50):
    print(f"  >{t}%: {np.sum(r > t)} ECGs ({np.mean(r > t)*100:.1f}%)")
k = max(1, int(round(len(r) * 0.05)))
top = np.sort(r)[::-1]
print(f"  Contribution of the top 5% ({k} ECGs) to the mean: {top[:k].sum() / r.sum() * 100:.0f}%")
print(f"  Mean excluding the top 5% (reference only): {np.sort(r)[:-k].mean():.1f}%")

# Recording with the poorest cancellation (highest residue rate / lowest suppression ratio)
paths_v = metadata.loc[valid, 'ecg_path'].values
i_rate, i_supp = np.nanargmax(rate_v), np.nanargmin(supp_v)
print(f"\nHighest residue rate   : {os.path.basename(paths_v[i_rate])} "
      f"(residue {rate_v[i_rate]:.0f}%, suppression ratio {supp_v[i_rate]:.2f})")
print(f"Lowest suppression ratio: {os.path.basename(paths_v[i_supp])} "
      f"(residue {rate_v[i_supp]:.0f}%, suppression ratio {supp_v[i_supp]:.2f})")


In [ ]:
# Group comparison of DF
print("=== Group comparison of DF ===\n")
for nm, m in [("Termination", y_v==1), ("Non-termination", y_v==0)]:
    d = DF_v[m]
    q1, q2, q3 = np.percentile(d, [25, 50, 75])
    print(f"{nm:16s}: median {q2:.2f} Hz (IQR {q1:.2f}–{q3:.2f}), range {d.min():.2f}–{d.max():.2f}, n={m.sum()}")

# Effect size: median difference (patient-level cluster bootstrap)
stat_median = lambda idx: (np.median(DF_v[idx][y_v[idx]==1])
                           - np.median(DF_v[idx][y_v[idx]==0]))
d, lo, hi, _, _ = cluster_bootstrap(y_v, pid_v, stat_median)
print(f"\nMedian difference (termination − non-termination): {d:+.2f} Hz [95% CI {lo:+.2f}, {hi:+.2f}]")

# Main inference: univariable AUROC (prespecified direction: lower DF indicates termination)
a, alo, ahi, p_auc = cluster_bootstrap_auc(y_v, -DF_v, pid_v)
print(f"Univariable AUROC: {a:.3f} [95% CI {alo:.3f}, {ahi:.3f}], p≈{p_auc:.4f}")

# Reference (assumes independence)
print(f"\nReference (Mann-Whitney U test): p = {mannwhitneyu(DF_v[y_v==1], DF_v[y_v==0])[1]:.4f}")

In [ ]:
# Decomposing the difference in p values between methods
# Treating each ECG as its own cluster gives an ordinary bootstrap assuming independence
ecg_ids = np.arange(len(y_v))
print("=== Independence assumption (ECG level) vs patient-level clusters ===\n")
for lab, g in [("ECG level (independence)", ecg_ids), ("Patient-level clusters", pid_v)]:
    d, lo, hi, p_med, _ = cluster_bootstrap(y_v, g, stat_median)
    a, alo, ahi, p_auc  = cluster_bootstrap_auc(y_v, -DF_v, g)
    print(f"{lab}")
    print(f"  Median difference {d:+.2f} Hz [{lo:+.2f}, {hi:+.2f}], p≈{p_med:.4f}")
    print(f"  AUROC    {a:.3f} [{alo:.3f}, {ahi:.3f}], p≈{p_auc:.4f}")

In [ ]:
# Univariable AUROC of clinical variables (direction stated)
print("=== Univariable AUROC ===\n")
for nm, sc, note in [("DF (lower = termination)", -DF_v, "prespecified"),
                     ("LAD (lower = termination)", -LAD_v, ""),
                     ("Age (lower = termination)", -Age_v, "")]:
    a, lo, hi, p = cluster_bootstrap_auc(y_v, sc, pid_v)
    print(f"{nm:28s}: {a:.3f} [{lo:.3f}, {hi:.3f}], p≈{p:.4f}  {note}")

print("\n=== Distribution of clinical variables ===")
for nm, v in [("LAD", LAD_v), ("Age", Age_v)]:
    print(f"{nm}: termination {np.median(v[y_v==1]):.1f} / non-termination {np.median(v[y_v==0]):.1f}"
          f" (Mann-Whitney p={mannwhitneyu(v[y_v==1], v[y_v==0])[1]:.3f})")

# Correlation between DF and clinical variables
print("\n=== Correlation (Spearman) ===")
for nm, v in [("Age", Age_v), ("LAD", LAD_v)]:
    rho, p = spearmanr(DF_v, v)
    print(f"DF vs {nm}: rho = {rho:+.3f}, p = {p:.4f}")

In [ ]:
# Recordings with DF at the lower band limit (3.5 Hz)
low = DF_v <= 3.55
print(f"DF at the lower limit: {low.sum()} ECGs "
      f"(termination {np.sum(low & (y_v==1))}/{np.sum(y_v==1)}, "
      f"non-termination {np.sum(low & (y_v==0))}/{np.sum(y_v==0)}), "
      f"{len(np.unique(pid_v[low]))} patients")

## 5. Primary analysis (prediction models)

Primary model (prespecified): DF + LAD + age
Secondary models (prespecified): DF alone / LAD alone / DF + LAD / DF + ECG-FM

All preprocessing is fitted on the training folds only.

In [ ]:
def get_oof(X_spec, y, groups, n_splits=N_SPLITS, seed=SEED,
            C=C_PARAM, n_pca=N_PCA, grouped=True):
    '''Out-of-fold predictions for one repetition

    X_spec : feature matrix, or a tuple ('embedding', DF, embedding matrix)
    grouped: if True, patient IDs are used as groups (ECG-level analysis)
    '''
    if grouped:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = cv.split(np.zeros(len(y)), y, groups=groups)
    else:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = cv.split(X_spec if not isinstance(X_spec, tuple) else np.zeros(len(y)), y)

    oof = np.zeros(len(y))
    is_emb = isinstance(X_spec, tuple)

    for tr, te in splits:
        if is_emb:
            _, dfv, emb = X_spec
            es = StandardScaler()
            emb_tr = es.fit_transform(emb[tr]); emb_te = es.transform(emb[te])
            pca = PCA(n_components=n_pca, random_state=seed)
            emb_tr = pca.fit_transform(emb_tr); emb_te = pca.transform(emb_te)
            if dfv is None:                      # embedding only
                X_tr, X_te = emb_tr, emb_te
            else:
                ds = StandardScaler()
                df_tr = ds.fit_transform(dfv[tr].reshape(-1,1))
                df_te = ds.transform(dfv[te].reshape(-1,1))
                X_tr = np.hstack([df_tr, emb_tr]); X_te = np.hstack([df_te, emb_te])
        else:
            sc = StandardScaler()
            X_tr = sc.fit_transform(X_spec[tr]); X_te = sc.transform(X_spec[te])

        clf = LogisticRegression(C=C, class_weight='balanced', max_iter=1000)
        clf.fit(X_tr, y[tr])
        oof[te] = clf.predict_proba(X_te)[:, 1]
    return oof


def repeated_auroc(X_spec, y, groups, n_repeats=N_REPEATS, seed0=SEED, grouped=True):
    '''AUROC of repeated cross-validation'''
    return np.array([roc_auc_score(y, get_oof(X_spec, y, groups, seed=seed0+r,
                                              grouped=grouped))
                     for r in range(n_repeats)])

In [ ]:
# Feature sets
X_dict = {
    'DF alone':          DF_v.reshape(-1, 1),
    'LAD alone':         LAD_v.reshape(-1, 1),
    'DF+LAD':          np.column_stack([DF_v, LAD_v]),
    'DF+LAD+age (primary)':  np.column_stack([DF_v, LAD_v, Age_v]),
}

# ECG-FM embedding (extracted in 01_ECG-FM_extraction.ipynb)
EMB_DIR = os.path.join(ROOT, "ecgfm")  # output of 01_ECG-FM_extraction.ipynb
EMB_PATH = os.path.join(EMB_DIR, "emb_transformer_full.npy")
try:
    emb_raw = np.load(EMB_PATH)

    # Alignment check: IDs, labels (and paths) saved at extraction must match the mapping table (before de-duplication)
    n_raw = len(KEEP)
    ok_align = emb_raw.shape[0] == n_raw
    pid_path = os.path.join(EMB_DIR, "patient_ids.npy")
    lab_path = os.path.join(EMB_DIR, "labels.npy")
    path_path = os.path.join(EMB_DIR, "ecg_paths.npy")
    if ok_align and os.path.exists(pid_path) and os.path.exists(lab_path):
        pid_e = np.load(pid_path, allow_pickle=True)[KEEP.values]
        lab_e = np.load(lab_path).astype(int)[KEEP.values]
        ok_align &= bool(np.all(pid_e.astype(str) == patient_ids.astype(str))
                         and np.all(lab_e == labels))
    if ok_align and os.path.exists(path_path):
        p_e = np.load(path_path, allow_pickle=True)[KEEP.values]
        ok_align &= bool(np.all([os.path.basename(str(a)) == os.path.basename(str(b))
                                 for a, b in zip(p_e, metadata['ecg_path'])]))
    print(f"Alignment of embeddings with the mapping table: {'OK' if ok_align else 'MISMATCH'}")

    if ok_align:
        emb_all = emb_raw[KEEP.values]
        X_dict['DF+ECG-FM'] = ('embedding', DF_v, emb_all[valid])
        X_dict['ECG-FM alone (ref)'] = ('embedding', None, emb_all[valid])
        print(f"ECG-FM embedding: {emb_all[valid].shape}")
    else:
        print("Warning: embeddings do not match the mapping table; ECG-FM models are skipped")
except FileNotFoundError:
    print("ECG-FM embedding not found; ECG-FM models are skipped")


In [ ]:
print("=== Primary analysis: AUROC by model ===\n")
print(f"{'Model':24s} {'AUROC':>7s} {'95%CI':>18s} {'p':>8s} {'SD':>8s}")
print("-" * 70)

oof_store, auc_store = {}, {}
for name, X_spec in X_dict.items():
    oofs = np.array([get_oof(X_spec, y_v, pid_v, seed=SEED + r) for r in range(N_REPEATS)])
    oof  = oofs.mean(axis=0)                                  # average the 10 repetitions for each ECG
    sd   = np.std([roc_auc_score(y_v, o) for o in oofs])      # reference: variation due to data splitting
    a, lo, hi, p = cluster_bootstrap_auc(y_v, oof, pid_v)
    oof_store[name] = oof
    auc_store[name] = (a, sd, lo, hi, p)
    mark = "*" if "primary" in name else " "
    print(f"{mark}{name:23s} {a:>7.3f}   [{lo:.3f}, {hi:.3f}] {p:>8.4f} {sd:>8.3f}")

print("\n* = prespecified primary model")
print("Discrimination is not established when the 95% CI includes 0.5")
print("AUROC and 95% CI are calculated from predictions averaged over 10 repetitions; SD is across repetitions")

## 6. Model comparison

The AUROCs of two models are calculated on the same bootstrap samples, and their difference is evaluated.
P values are nominal, without correction for multiple comparisons.

In [ ]:
def compare_models(y, prob1, prob2, groups, n_boot=N_BOOT, seed=SEED):
    '''Observed AUROC difference, 95% CI and two-sided p value'''
    stat = lambda idx: (roc_auc_score(y[idx], prob1[idx])
                        - roc_auc_score(y[idx], prob2[idx]))
    return cluster_bootstrap(y, groups, stat, n_boot, seed)[:4]


def delong_test(y, prob1, prob2):
    '''DeLong test (reference only; assumes independent observations)'''
    y = np.asarray(y)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    n1, n0 = len(pos), len(neg)

    def structural(prob):
        X = np.asarray(prob)[pos][:, None]
        Y = np.asarray(prob)[neg][None, :]
        psi = (X > Y).astype(float) + 0.5 * (X == Y)
        return psi.mean(), psi.mean(axis=1), psi.mean(axis=0)

    a1, v10_1, v01_1 = structural(prob1)
    a2, v10_2, v01_2 = structural(prob2)
    S10 = np.cov(np.vstack([v10_1, v10_2]))
    S01 = np.cov(np.vstack([v01_1, v01_2]))
    var = (S10[0,0]+S10[1,1]-2*S10[0,1])/n1 + (S01[0,0]+S01[1,1]-2*S01[0,1])/n0
    if var <= 0:
        return a1, a2, np.nan, np.nan
    z = (a1 - a2) / np.sqrt(var)
    return a1, a2, z, 2*(1 - norm.cdf(abs(z)))


COMPARISONS = [
    ("DF alone vs LAD alone",       'DF alone', 'LAD alone'),
    ("DF alone vs primary model",   'DF alone', 'DF+LAD+age (primary)'),
    ("DF alone vs DF+LAD",          'DF alone', 'DF+LAD'),
    ("Primary model vs LAD alone",  'DF+LAD+age (primary)', 'LAD alone'),
]

In [ ]:
print("=== Model comparison ===\n")
print(f"{'Comparison':28s} {'ΔAUROC':>8s} {'95%CI':>20s} {'p(BS)':>8s} {'p(DeLong)':>11s}")
print("-" * 76)

for name, k1, k2 in COMPARISONS:
    d, lo, hi, p_bs = compare_models(y_v, oof_store[k1], oof_store[k2], pid_v)
    _, _, _, p_dl = delong_test(y_v, oof_store[k1], oof_store[k2])
    sig = "*" if (lo > 0 or hi < 0) else " "
    print(f"{name:28s} {d:>+8.3f} [{lo:>+7.3f}, {hi:>+7.3f}] {p_bs:>8.4f}{sig} {p_dl:>11.4f}")

print("\n* 95% CI excludes 0 (nominal; no correction for multiple comparisons)")
print("The DeLong test assumes independent observations and is shown for reference only")

## 7. Sensitivity analyses

**7-1 (prespecified)**: to evaluate the effect of multiple ECGs per patient, the ECG closest to surgery is used for each patient.
Stratified k-fold cross-validation is used because each patient contributes one sample.

**7-2 (post hoc)**: univariable AUROC of DF after excluding the recording with the lowest suppression ratio (< 2).

In [ ]:
meta_v = metadata[valid].reset_index(drop=True).copy()
meta_v['DF'] = DF_v
meta_v['label'] = y_v

idx_near = meta_v.groupby('ID')['Duration'].idxmin()
dp = meta_v.loc[idx_near].reset_index(drop=True)

DF_pt  = dp['DF'].values.astype(float)
LAD_pt = dp['LAD'].values.astype(float)
Age_pt = dp['Age'].values.astype(float)
y_pt   = dp['label'].values.astype(int)
pid_pt = dp['ID'].values

print(f"Patient level: {len(dp)} patients")
print("Labels:", dict(zip(*np.unique(y_pt, return_counts=True))))
print(f"Days from selected ECG to surgery: median {np.median(dp['Duration']):.0f}, "
      f"range {dp['Duration'].min():.0f}–{dp['Duration'].max():.0f}\n")

print("=== Group comparison of DF from the ECG closest to surgery ===")
for nm, m in [("Termination", y_pt==1), ("Non-termination", y_pt==0)]:
    q1, q2, q3 = np.percentile(DF_pt[m], [25, 50, 75])
    print(f"{nm:16s}: median {q2:.2f} Hz (IQR {q1:.2f}–{q3:.2f}), n={m.sum()}")

stat_med_pt = lambda idx: (np.median(DF_pt[idx][y_pt[idx]==1])
                           - np.median(DF_pt[idx][y_pt[idx]==0]))
d, lo, hi, _, _ = cluster_bootstrap(y_pt, pid_pt, stat_med_pt)
print(f"\nMedian difference: {d:+.2f} Hz [95% CI {lo:+.2f}, {hi:+.2f}]")

a, alo, ahi, p_auc = cluster_bootstrap_auc(y_pt, -DF_pt, pid_pt)
print(f"Univariable AUROC: {a:.3f} [{alo:.3f}, {ahi:.3f}], p≈{p_auc:.4f}")
print(f"\nMann-Whitney U test: p = {mannwhitneyu(DF_pt[y_pt==1], DF_pt[y_pt==0])[1]:.4f}")

In [ ]:
X_pt = {
    'DF alone':          DF_pt.reshape(-1,1),
    'LAD alone':         LAD_pt.reshape(-1,1),
    'DF+LAD':          np.column_stack([DF_pt, LAD_pt]),
    'DF+LAD+age (primary)':  np.column_stack([DF_pt, LAD_pt, Age_pt]),
}

print("=== Sensitivity analysis: AUROC by model (patient level) ===\n")
print(f"{'Model':24s} {'AUROC':>7s} {'95%CI':>18s} {'p':>8s} {'SD':>8s}")
print("-" * 70)
for name, X in X_pt.items():
    oofs = np.array([get_oof(X, y_pt, pid_pt, seed=SEED + r, grouped=False)
                     for r in range(N_REPEATS)])
    oof  = oofs.mean(axis=0)
    sd   = np.std([roc_auc_score(y_pt, o) for o in oofs])
    a, lo, hi, p = cluster_bootstrap_auc(y_pt, oof, pid_pt)
    mark = "*" if "primary" in name else " "
    print(f"{mark}{name:23s} {a:>7.3f}   [{lo:.3f}, {hi:.3f}] {p:>8.4f} {sd:>8.3f}")

In [ ]:
# 7-2 (post hoc): exclude the recording with the lowest suppression ratio (< 2)
keep = ~(supp_v < 2)
print(f"Excluded: {(~keep).sum()} ECG(s) -> {keep.sum()} ECGs / {len(np.unique(pid_v[keep]))} patients")
for nm, m in [("Termination", keep & (y_v==1)), ("Non-termination", keep & (y_v==0))]:
    q1, q2, q3 = np.percentile(DF_v[m], [25, 50, 75])
    print(f"{nm:16s}: median {q2:.2f} Hz (IQR {q1:.2f}–{q3:.2f})")
a, lo, hi, p = cluster_bootstrap_auc(y_v[keep], -DF_v[keep], pid_v[keep])
print(f"Univariable AUROC: {a:.3f} [95% CI {lo:.3f}, {hi:.3f}], p≈{p:.4f}")

## 8. Sample size estimation

Using the Hanley-McNeil method, the number of patients required to distinguish the univariable AUROC of DF in the
patient-level analysis (7-1) from 0.5 is estimated (two-sided α = 0.05, power 0.80).
The case ratio of this cohort is used; success rates of 60–80% are also shown for reference.

In [ ]:
def n_for_auc(auc, ratio, alpha=0.05, power=0.80):
    '''Hanley-McNeil method: sample size to distinguish the AUROC from 0.5 (ratio = negatives/positives)'''
    z_a, z_b = norm.ppf(1 - alpha/2), norm.ppf(power)
    q1, q2 = auc / (2 - auc), 2 * auc**2 / (1 + auc)
    n_pos = 2
    while True:
        n_neg = n_pos * ratio
        v0 = (n_pos + n_neg + 1) / (12 * n_pos * n_neg)            # variance under the null hypothesis (AUC = 0.5)
        v1 = (auc*(1-auc) + (n_pos-1)*(q1-auc**2)
              + (n_neg-1)*(q2-auc**2)) / (n_pos * n_neg)           # variance under the assumed AUC
        if z_a*np.sqrt(v0) + z_b*np.sqrt(v1) <= auc - 0.5:
            return n_pos, int(np.ceil(n_neg))
        n_pos += 1

AUC_PT = roc_auc_score(y_pt, -DF_pt)          # patient-level univariable AUROC of DF
n_pos_obs = int(np.sum(y_pt == 1))
n_neg_obs = int(np.sum(y_pt == 0))
ratio = n_neg_obs / n_pos_obs

npos, nneg = n_for_auc(AUC_PT, ratio)
print(f"Effect size: AUROC {AUC_PT:.3f} (patient level)")
print(f"Ratio of this cohort (termination {n_pos_obs} : non-termination {n_neg_obs}): "
      f"termination {npos}, non-termination {nneg}, total {npos+nneg}\n")

print(f"{'Success':>8s} {'Term.':>8s} {'Non-term.':>10s} {'Total':>8s}")
print("-" * 36)
for s in [0.60, 0.70, 0.80]:
    npos, nneg = n_for_auc(AUC_PT, (1 - s) / s)
    print(f"{s*100:>7.0f}% {npos:>8d} {nneg:>10d} {npos+nneg:>8d}")

## 9. Post hoc exploratory analyses

These analyses were not prespecified and are not corrected for multiple comparisons; they are hypothesis-generating.

1. f-wave amplitude (lead V1 as the main value; lead II for reference)
2. DF in the 12 leads
3. Window function (Hamming) and spectral smoothing
4. AF type and DF
5. Within-patient variability of DF and averaging of spectra across ECGs

### 9-1. f-wave amplitude

Following clinical practice and previous reports, f-wave amplitude in **lead V1** is presented as the main value; lead II is shown for reference.


In [ ]:
def f_wave_amplitude(csv_path, lead=LEAD, fs=FS):
    '''f-wave amplitude: ratio to QRS amplitude [%], robust P-P [μV], near-max P-P [μV]'''
    sig = preprocess(csv_path, lead, fs)
    if sig is None:
        return np.nan, np.nan, np.nan
    r_peaks = detect_r_peaks(sig, fs)
    residual, n_used, status, internals = qrst_subtract(
        sig, r_peaks, fs, return_internals=True)
    if residual is None:
        return np.nan, np.nan, np.nan

    pre, post = internals['pre'], internals['post']
    mask = np.ones(len(residual), dtype=bool)
    for r in internals['valid_peaks']:
        mask[max(0, r-pre):min(len(residual), r+post)] = False
    f = residual[mask]
    if len(f) < fs // 2:
        return np.nan, np.nan, np.nan

    pp_rob = np.percentile(f, 97.5) - np.percentile(f, 2.5)
    pp_max = np.percentile(f, 99.5) - np.percentile(f, 0.5)
    qrs_pp = np.median([t.max()-t.min() for t in internals['templates'].values()])
    ratio = pp_rob / qrs_pp * 100 if qrs_pp > 1e-9 else np.nan
    return ratio, pp_rob*UV_PER_COUNT, pp_max*UV_PER_COUNT


amp_rows = [f_wave_amplitude(p) for p in tqdm(metadata['ecg_path'], desc="f-wave amplitude")]
amp_ratio = np.array([a[0] for a in amp_rows])[valid]
amp_rob   = np.array([a[1] for a in amp_rows])[valid]
amp_max   = np.array([a[2] for a in amp_rows])[valid]

ok = ~np.isnan(amp_max)
print(f"\nCalculable: {ok.sum()} ECGs / {len(np.unique(pid_v[ok]))} patients")
if (~ok).sum():
    print(f"Heart rate of the {(~ok).sum()} non-calculable ECGs: median {np.median(hr_v[~ok]):.0f} /min")
print(f"\nf-wave amplitude (ratio to QRS): median {np.nanmedian(amp_ratio):.1f} %")
print(f"f-wave amplitude (robust P-P)  : median {np.median(amp_rob[ok]):.0f} μV")
print(f"f-wave amplitude (near-max P-P): median {np.median(amp_max[ok]):.0f} μV")

THRESH_UV = 100.0   # 0.1 mV (boundary between coarse and fine AF)
coarse = (amp_max[ok] >= THRESH_UV)
print(f"  Coarse AF (≥0.1 mV): {coarse.sum()} ECGs ({coarse.mean()*100:.0f}%)")

a, lo, hi, p = cluster_bootstrap_auc(y_v[ok], amp_max[ok], pid_v[ok])
print(f"\nf-wave amplitude, univariable AUROC: {a:.3f} [{lo:.3f}, {hi:.3f}], p≈{p:.4f}")

#### f-wave amplitude in lead V1 (main value)

At our institution, the indication for the maze procedure takes the f-wave amplitude in lead V1 (0.1 mV) into account, so the degree of range restriction is assessed in lead V1.
For comparison with the Fushimi AF Registry (Kawaji et al., 2022), one ECG per patient (closest to surgery) is used.


In [ ]:
amp_rows_v1 = [f_wave_amplitude(p, lead='V1') for p in tqdm(metadata['ecg_path'], desc="f-wave amplitude (V1)")]
amp_rob_v1 = np.array([a[1] for a in amp_rows_v1])[valid]
amp_max_v1 = np.array([a[2] for a in amp_rows_v1])[valid]

ok1 = ~np.isnan(amp_max_v1)
print(f"V1 available: {ok1.sum()} ECGs / {len(np.unique(pid_v[ok1]))} patients")
q1, q2, q3 = np.percentile(amp_max_v1[ok1], [25, 50, 75])
print(f"V1 f-wave amplitude (near-max P-P): median {q2:.0f} μV (IQR {q1:.0f}–{q3:.0f})")
print(f"  coarse AF (≥0.1 mV): {np.sum(amp_max_v1[ok1] >= THRESH_UV)} ECGs "
      f"({np.mean(amp_max_v1[ok1] >= THRESH_UV)*100:.0f}%)")

# Patient level (ECG closest to surgery)
near_v1 = metadata.loc[valid].reset_index(drop=True)
idx_pt_v1 = near_v1.loc[ok1].groupby('ID')['Duration'].idxmin().values
c_pt = amp_max_v1[idx_pt_v1] >= THRESH_UV
print(f"  Patient level (closest to surgery): {c_pt.sum()} / {len(c_pt)} ({c_pt.mean()*100:.0f}%)")

# More conservative measure (robust P-P)
print(f"\nV1 robust P-P: median {np.median(amp_rob_v1[ok1]):.0f} μV, "
      f"coarse {np.mean(amp_rob_v1[ok1] >= THRESH_UV)*100:.0f}%, "
      f"patient level {np.mean(amp_rob_v1[idx_pt_v1] >= THRESH_UV)*100:.0f}%")

a, lo, hi, p = cluster_bootstrap_auc(y_v[ok1], amp_max_v1[ok1], pid_v[ok1])
print(f"\nV1 f-wave amplitude, univariable AUROC: {a:.3f} [{lo:.3f}, {hi:.3f}], p≈{p:.4f}")


### 9-2. DF in the 12 leads

In [ ]:
lead_rows = []
for p in tqdm(metadata['ecg_path'], desc="12-lead DF"):
    lead_rows.append({L: compute_df(p, lead=L) for L in LEADS_12})
df_leads = pd.DataFrame(lead_rows)[valid.tolist()].reset_index(drop=True)

print("\n=== DF by lead ===")
print(f"{'Lead':6s} {'n':>5s} {'Median':>8s} {'DF<5Hz':>9s} {'AUROC':>8s} {'95%CI':>18s}")
print("-" * 62)
for L in LEADS_12:
    x = df_leads[L].values
    m = ~np.isnan(x)
    if m.sum() < 50:
        print(f"{L:6s} too few calculable ECGs"); continue
    a, lo, hi, _ = cluster_bootstrap_auc(y_v[m], -x[m], pid_v[m])
    mark = "*" if L == LEAD else " "
    print(f"{mark}{L:5s} {m.sum():>5d} {np.median(x[m]):>8.2f} {np.mean(x[m]<5.0)*100:>8.0f}% "
          f"{a:>8.3f}   [{lo:.3f}, {hi:.3f}]")
print("\n* = prespecified lead; no correction for multiple comparisons")

In [ ]:
print("=== Correlation of DF between lead pairs ===\n")
pairs = [('aVF','V1'), ('V1','V5'), ('Ⅱ','aVF'), ('Ⅱ','V1'), ('Ⅱ','Ⅰ')]
for a_, b_ in pairs:
    x, y_ = df_leads[a_], df_leads[b_]
    m = x.notna() & y_.notna()
    print(f"{a_:4s}–{b_:4s}: Pearson r = {pearsonr(x[m], y_[m])[0]:.3f}, "
          f"Spearman rho = {spearmanr(x[m], y_[m])[0]:.3f}, n = {m.sum()}")
print("\n(Bollmann 1998: aVF–V1 r=0.97, V1–V5 r=0.84)")

# Ratio of the mean amplitude below the DF search band to that within the band (low DF in lateral leads)
def spectral_tilt(csv_path, lead, fs=FS):
    sig = preprocess(csv_path, lead, fs)
    if sig is None:
        return np.nan
    residual, _, st = qrst_subtract(sig, detect_r_peaks(sig, fs), fs)
    if residual is None:
        return np.nan
    fr, fv, _ = spectrum(residual, fs)
    return fv[(fr >= 0.5) & (fr < AF_BAND[0])].mean() / (
        fv[(fr >= AF_BAND[0]) & (fr <= AF_BAND[1])].mean() + 1e-12)

tilt = pd.DataFrame({L: [spectral_tilt(p, L) for p in metadata['ecg_path']]
                     for L in tqdm(LEADS_12, desc="Low-frequency ratio")})[valid.tolist()].reset_index(drop=True)

print("\n=== Low-frequency ratio (0.5–3.5 Hz / 3.5–9.0 Hz) ===")
for L in LEADS_12:
    q1, q2, q3 = np.nanpercentile(tilt[L], [25, 50, 75])
    print(f"  {L:4s}: {q2:.2f} ({q1:.2f}–{q3:.2f}), n = {tilt[L].notna().sum()}")

### 9-3. Effect of processing conditions (window function and smoothing)

In [ ]:
# Window function (Hamming)
DF_ham = np.array([compute_df(p, window=np.hamming)
                   for p in tqdm(metadata['ecg_path'], desc="Hamming window")])[valid]

# Spectral smoothing (0.25 Hz width)
DF_sm  = np.array([compute_df(p, smooth_hz=0.25)
                   for p in tqdm(metadata['ecg_path'], desc="Smoothing")])[valid]
DF_sm_ham = np.array([compute_df(p, window=np.hamming, smooth_hz=0.25)
                      for p in tqdm(metadata['ecg_path'], desc="Smoothing + Hamming")])[valid]

print("\n=== Effect of the window function ===")
print(f"Hanning vs Hamming: difference ≥0.1 Hz in {np.sum(np.abs(DF_v-DF_ham)>0.1)} ECGs "
      f"({np.sum(np.abs(DF_v-DF_ham)>0.1)/len(DF_v)*100:.1f}%), "
      f"Spearman rho = {spearmanr(DF_v, DF_ham)[0]:.3f}")
a, lo, hi, p = cluster_bootstrap_auc(y_v, -DF_ham, pid_v)
print(f"  AUROC with Hamming window: {a:.3f} [{lo:.3f}, {hi:.3f}], "
      f"Mann-Whitney p = {mannwhitneyu(DF_ham[y_v==1], DF_ham[y_v==0])[1]:.4f}")

print("\n=== Effect of spectral smoothing ===")
print(f"Difference between windows after smoothing: {np.sum(np.abs(DF_sm-DF_sm_ham)>0.1)} ECGs "
      f"({np.sum(np.abs(DF_sm-DF_sm_ham)>0.1)/len(DF_v)*100:.1f}%) "
      f"(before smoothing {np.sum(np.abs(DF_v-DF_ham)>0.1)}), "
      f"Spearman rho = {spearmanr(DF_sm, DF_sm_ham)[0]:.3f}")
print(f"Agreement with unsmoothed DF: Spearman rho = {spearmanr(DF_v, DF_sm)[0]:.3f}, "
      f"difference ≥0.1 Hz in {np.sum(np.abs(DF_v-DF_sm)>0.1)} ECGs")
a, lo, hi, p = cluster_bootstrap_auc(y_v, -DF_sm, pid_v)
print(f"  AUROC with smoothing: {a:.3f} [{lo:.3f}, {hi:.3f}], "
      f"Mann-Whitney p = {mannwhitneyu(DF_sm[y_v==1], DF_sm[y_v==0])[1]:.4f}")

### 9-4. AF type and DF

AF type is obtained from the patient table (Excel with a three-row header).
Tests use the ECG closest to surgery for each patient (as in 7-1), so observations are independent.

In [ ]:
PT_PATH = os.path.join(ROOT, "data/table.xlsx")  # <- path to the patient table

raw = pd.read_excel(PT_PATH, header=None)
ptab = raw.iloc[3:].reset_index(drop=True)
ptab = ptab[ptab[0].notna()].reset_index(drop=True)
ptab = pd.DataFrame({
    'ID':     'P' + ptab[0].astype(int).astype(str).str.zfill(4),   # <- adjust if the ID format differs
    'Age_pt': pd.to_numeric(ptab[1], errors='coerce'),
    'LAD_pt': pd.to_numeric(ptab[28], errors='coerce'),
    'AFtype': ptab[22].map({'発作性(<7d)': 0, '持続性(>7d)': 1, '長期持続性(>1y)': 2}),
    # source-table labels: paroxysmal (<7 d) = 0, persistent (>7 d) = 1, long-standing persistent (>1 y) = 2
})

# --- (1) Check the ID matching (age and LAD agree between tables) ---
chk = meta_v.groupby('ID')[['Age', 'LAD']].first().reset_index().merge(ptab, on='ID', how='left')
bad = chk[(chk['Age'] != chk['Age_pt']) | ((chk['LAD'] - chk['LAD_pt']).abs() > 0.5)]
print(f"Matched: {chk['AFtype'].notna().sum()}/{len(chk)} patients, age/LAD mismatch: {len(bad)}")
if len(bad):
    print(bad.head())

AF_v = meta_v['ID'].map(ptab.set_index('ID')['AFtype']).values
labels_af = {0: 'Paroxysmal', 1: 'Persistent', 2: 'Long-standing persistent'}

# --- (2) DF by AF type ---
print("\n=== DF by AF type (ECG level) ===")
for k, nm in labels_af.items():
    d = DF_v[AF_v == k]
    if len(d):
        q1, q2, q3 = np.percentile(d, [25, 50, 75])
        print(f"{nm:25s}: {q2:.2f} ({q1:.2f}–{q3:.2f}) Hz, n={len(d)} ECGs")

# Tests at the patient level (ECG closest to surgery): observations are independent
dp2 = dp.merge(ptab[['ID', 'AFtype']], on='ID', how='left')
print("\n=== DF by AF type (patient level, ECG closest to surgery) ===")
af_groups = []
for k, nm in labels_af.items():
    d = dp2.loc[dp2['AFtype'] == k, 'DF'].values
    af_groups.append(d)
    if len(d):
        q1, q2, q3 = np.percentile(d, [25, 50, 75])
        print(f"{nm:25s}: median {q2:.2f} Hz (IQR {q1:.2f}–{q3:.2f}), n={len(d)} patients")
print(f"Kruskal-Wallis: p = {kruskal(*[g for g in af_groups if len(g)])[1]:.4f}")
rho, p = spearmanr(dp2['AFtype'], dp2['DF'])
print(f"AF type (ordinal) vs DF, Spearman: rho = {rho:+.3f}, p = {p:.4f}")

# --- (3) Discrimination by DF within each AF type ---
print("\n=== Univariable AUROC of DF within each AF type (ECG level) ===")
for k in [1, 2]:
    m = AF_v == k
    n1, n0 = np.sum(y_v[m] == 1), np.sum(y_v[m] == 0)
    if n1 and n0:
        a, lo, hi, _ = cluster_bootstrap_auc(y_v[m], -DF_v[m], pid_v[m])
        print(f"{labels_af[k]:25s}: AUROC {a:.3f} [{lo:.3f}, {hi:.3f}]"
              f" (termination {n1} / non-termination {n0} ECGs, {len(np.unique(pid_v[m]))} patients)")

# Reference: discrimination by AF type itself (long-standing persistent = non-termination)
a, lo, hi, _ = cluster_bootstrap_auc(y_v, -(AF_v == 2).astype(float), pid_v)
print(f"\nAF type (long-standing persistent or not), univariable AUROC: {a:.3f} [{lo:.3f}, {hi:.3f}]")

### 9-5. Within-patient variability of DF and averaging of spectra across ECGs

In patients with multiple ECGs, we examine (1) within-patient variability of DF, (2) the relation between the difference in DF and the interval between recordings,
and (3) multimodality and discrimination when spectra of ECGs recorded on different days are averaged.
Each spectrum is normalized by its power within the search band before averaging.
Multimodality is assessed by the ratio of the second to the first peak (prominence ≥ 10% of the maximum; exploratory measure).


In [ ]:
dur_v = metadata.loc[valid, 'Duration'].values.astype(float)

def band_spectrum(path):
    sig = preprocess(path, LEAD, FS)
    if sig is None:
        return None
    res, _, _ = qrst_subtract(sig, detect_r_peaks(sig, FS), FS)
    if res is None:
        return None
    fr, fv, m = spectrum(res, FS)
    s = fv[m]
    return fr[m], s / s.sum()

def second_peak_ratio(s):
    pk, _ = find_peaks(s, prominence=0.1 * s.max())
    if len(pk) < 2:
        return 0.0
    v = np.sort(s[pk])[::-1]
    return v[1] / v[0]

specs = [band_spectrum(p) for p in tqdm(paths_v, desc="spectra")]
f_grid = next(s for s in specs if s is not None)[0]

rows = []
for p in np.unique(pid_v):
    idx = [i for i in np.where(pid_v == p)[0] if specs[i] is not None]
    if not idx:
        continue
    S = np.array([specs[i][1] for i in idx])
    avg = S.mean(axis=0)
    near = idx[int(np.argmin(dur_v[idx]))]
    rows.append(dict(pid=p, y=y_v[idx[0]], n=len(idx),
                     df_near=DF_v[near], df_avg=f_grid[np.argmax(avg)],
                     sd_within=np.std(DF_v[idx]) if len(idx) > 1 else np.nan,
                     ratio_single=np.mean([second_peak_ratio(s) for s in S]),
                     ratio_avg=second_peak_ratio(avg)))
spt = pd.DataFrame(rows)
multi = (spt['n'] >= 2).values

print(f"{len(spt)} patients ({multi.sum()} with multiple ECGs: "
      f"termination {(multi & (spt.y==1)).sum()} / non-termination {(multi & (spt.y==0)).sum()})")
sd = spt.loc[multi, 'sd_within']
print(f"\nWithin-patient SD of DF: median {sd.median():.2f} Hz (IQR {sd.quantile(.25):.2f}–{sd.quantile(.75):.2f})")
print(f"Between-patient SD of DF (closest to surgery): {spt['df_near'].std():.2f} Hz")

# Difference in DF vs interval between recordings (all within-patient pairs)
pp = []
for p in np.unique(pid_v):
    idx = np.where(pid_v == p)[0]
    for a_ in range(len(idx)):
        for b_ in range(a_ + 1, len(idx)):
            i, j = idx[a_], idx[b_]
            pp.append((p, abs(dur_v[i] - dur_v[j]), abs(DF_v[i] - DF_v[j])))
pp = pd.DataFrame(pp, columns=['pid', 'dday', 'ddf'])
rho, prho = spearmanr(pp['dday'], pp['ddf'])
print(f"\n{len(pp)} ECG pairs: |ΔDF| vs |Δdays| Spearman rho = {rho:+.3f}, p = {prho:.3f}")
for lo_, hi_ in [(0, 0), (1, 7), (8, 30), (31, 90), (91, 9999)]:
    d = pp[(pp.dday >= lo_) & (pp.dday <= hi_)]
    if len(d):
        per_pt = d.groupby('pid')['ddf'].median()
        print(f"  {lo_:>3}–{hi_:<4} days: {len(d):3d} pairs / {len(per_pt):2d} patients, "
              f"median |ΔDF| {d['ddf'].median():.2f} Hz, median of per-patient medians {per_pt.median():.2f} Hz")

print("\nMultimodality (second/first peak ratio, patients with multiple ECGs):")
print(f"  Individual spectra (mean per patient): median {spt.loc[multi, 'ratio_single'].median():.2f}")
print(f"  Averaged spectrum                    : median {spt.loc[multi, 'ratio_avg'].median():.2f}")

for name, m_ in [("All patients", np.ones(len(spt), bool)), ("Multiple ECGs only", multi)]:
    d = spt[m_]
    if d.y.nunique() < 2:
        continue
    print(f"\n[{name}, n={len(d)}] Univariable AUROC (lower DF = termination)")
    print(f"  DF closest to surgery  : {roc_auc_score(d.y, -d.df_near):.3f}")
    print(f"  DF of averaged spectrum: {roc_auc_score(d.y, -d.df_avg):.3f}")


In [ ]:
for lo_, hi_ in [(2, 2), (3, 3), (4, 99)]:
    d = spt[(spt.n >= lo_) & (spt.n <= hi_)]
    print(f"n={lo_}–{hi_}: {len(d)} patients, peak ratio individual {d.ratio_single.median():.2f} → averaged {d.ratio_avg.median():.2f}")


In [ ]:
pos = []
for p in np.unique(pid_v):
    for i in np.where(pid_v == p)[0]:
        if specs[i] is None: continue
        s = specs[i][1]
        pk, _ = find_peaks(s, prominence=0.1 * s.max())
        if len(pk) < 2: continue
        o = pk[np.argsort(s[pk])[::-1]][:2]
        f1, f2 = f_grid[o[0]], f_grid[o[1]]
        pos.append(max(f1, f2) / min(f1, f2))
pos = np.array(pos)
print(f"Recordings with >=2 peaks: {len(pos)}; higher/lower peak frequency ratio, median {np.median(pos):.2f}; "
      f"proportion within 1.9–2.1 (harmonic): {np.mean((pos > 1.9) & (pos < 2.1)):.0%}")

In [ ]:
# Post hoc check: zero padding (NFFT) and within-recording averaging (Welch)
from scipy.signal import periodogram, welch

BAND = (3.5, 9.0)

def residual(path):
    sig = preprocess(path, LEAD, FS)
    if sig is None:
        return None
    res, _, _ = qrst_subtract(sig, detect_r_peaks(sig, FS), FS)
    return res

def df_and_ratio(f, p):
    m = (f >= BAND[0]) & (f <= BAND[1])
    f, p = f[m], p[m]
    pk, _ = find_peaks(p, prominence=0.1 * p.max())
    r = 0.0 if len(pk) < 2 else np.sort(p[pk])[::-1][1] / np.sort(p[pk])[::-1][0]
    return f[np.argmax(p)], r

res_list = [residual(p) for p in tqdm(paths_v, desc="Residuals")]

methods = {
    'NFFT=signal length': lambda x: periodogram(x, FS, window='hann', nfft=len(x)),
    'Welch 4s/50%': lambda x: welch(x, FS, window='hann', nperseg=int(4 * FS),
                                    noverlap=int(2 * FS), nfft=65536),
}
out = pd.DataFrame({'pid': pid_v, 'y': y_v, 'dday': dur_v, 'DF_original': DF_v})
for name, fn in methods.items():
    v = [df_and_ratio(*fn(r)) if r is not None else (np.nan, np.nan) for r in res_list]
    out[f'DF_{name}'] = [a for a, _ in v]
    out[f'R_{name}'] = [b for _, b in v]

def summarize(col):
    d = out.dropna(subset=[col])
    sd = d.groupby('pid')[col].std().dropna()
    same = []
    for _, g in d.groupby('pid'):
        g0 = g.groupby('dday')[col]
        same += [abs(a - b) for _, s in g0 for i, a in enumerate(s.values) for b in s.values[i + 1:]]
    near = d.loc[d.groupby('pid')['dday'].idxmin()]
    return (f"within-patient SD median {sd.median():.2f} Hz | same-day |ΔDF| median {np.median(same):.2f} Hz | "
            f"AUROC ECG-level {roc_auc_score(d.y, -d[col]):.3f} / patient-level {roc_auc_score(near.y, -near[col]):.3f}")

for col in ['DF_original'] + [f'DF_{k}' for k in methods]:
    print(f"{col:<16} {summarize(col)}")
for k in methods:
    print(f"Peak ratio {k}: median {out[f'R_{k}'].median():.2f}")
print(f"DF difference >0.2 Hz between original and NFFT=signal length: {(abs(out['DF_original'] - out['DF_NFFT=signal length']) > 0.2).mean():.0%}")

In [ ]:
# A: subtraction quality vs multimodality
R = np.array([second_peak_ratio(s[1]) if s is not None else np.nan for s in specs])
m = ~np.isnan(R) & ~np.isnan(rate_v)
print(f"[A] peak ratio vs QRS residue rate rho={spearmanr(R[m], rate_v[m])[0]:+.2f}, "
      f"vs suppression ratio rho={spearmanr(R[m], supp_v[m])[0]:+.2f}")
print(f"    residue 0%: ratio median {np.median(R[m & (rate_v == 0)]):.2f} (n={np.sum(m & (rate_v == 0))}) / "
      f">0%: {np.median(R[m & (rate_v > 0)]):.2f} (n={np.sum(m & (rate_v > 0))})")

# B: peak spacing vs heart-rate frequency
gap = []
for s in specs:
    if s is None: gap.append(np.nan); continue
    pk, _ = find_peaks(s[1], prominence=0.1 * s[1].max())
    o = pk[np.argsort(s[1][pk])[::-1]][:2]
    gap.append(abs(f_grid[o[0]] - f_grid[o[1]]) if len(o) == 2 else np.nan)
gap = np.array(gap); hrhz = hr_v / 60
mb = ~np.isnan(gap) & ~np.isnan(hrhz)
print(f"[B] heart rate median {np.median(hr_v[mb]):.0f}/min ({np.median(hrhz[mb]):.2f} Hz), "
      f"peak spacing median {np.median(gap[mb]):.2f} Hz")
print(f"    spacing within ±0.15 Hz of heart-rate frequency: {np.mean(abs(gap[mb] - hrhz[mb]) < 0.15):.0%}, "
      f"rho={spearmanr(gap[mb], hrhz[mb])[0]:+.2f}")

# C: comparison with lead V1
def ratio_lead(path, lead):
    sig = preprocess(path, lead, FS)
    if sig is None: return np.nan
    res, _, _ = qrst_subtract(sig, detect_r_peaks(sig, FS), FS)
    if res is None: return np.nan
    fr, fv, mm = spectrum(res, FS)
    return second_peak_ratio(fv[mm] / fv[mm].sum())
RV1 = np.array([ratio_lead(p, 'V1') for p in tqdm(paths_v, desc="V1")])
print(f"[C] peak ratio median II {np.nanmedian(R):.2f} / V1 {np.nanmedian(RV1):.2f}")

## 10. Saving results and software versions

In [ ]:
import sys

results = pd.DataFrame({
    'ID': pid_v, 'label': y_v, 'DF': DF_v, 'LAD': LAD_v, 'Age': Age_v,
    'HR': hr_v, 'organization': org_v, 'suppression': supp_v,
    'qrs_residue_rate': rate_v,
    'DF_hamming': DF_ham, 'DF_smoothed': DF_sm,
    'f_amp_ratio': amp_ratio, 'f_amp_uV': amp_max,
})
results.to_csv(os.path.join(ROOT, 'analysis_results.csv'),
               index=False, encoding='utf-8-sig')
print(f"Saved: {os.path.join(ROOT, 'analysis_results.csv')}")
print(f"{len(results)} rows x {results.shape[1]} columns")

import sklearn, scipy, matplotlib
print(f"\nPython {sys.version.split()[0]} / numpy {np.__version__} / "
      f"pandas {pd.__version__} / scipy {scipy.__version__} / "
      f"scikit-learn {sklearn.__version__}")

# Libraries used to extract the ECG-FM embedding (shown only if installed in this environment)
try:
    import torch, fairseq_signals
    print(f"PyTorch {torch.__version__} / fairseq-signals {fairseq_signals.__version__}")
except ImportError:
    print("PyTorch / fairseq-signals not installed in this environment")

## Appendix: Figure 1 (representative example of ECG processing)

In [ ]:
# ============================================================
# Figure 1: representative example of ECG processing (lead II)
#   A: band-pass filtered ECG with detected QRS complexes
#   B: residual signal after QRST cancellation (f waves)
#   C: amplitude spectrum with the DF search band and DF
# Run after Section 2 (uses metadata, preprocess, detect_r_peaks, qrst_subtract, spectrum).
# ============================================================
import matplotlib.pyplot as plt

FIG_ID   = None   # patient ID for the representative example; None = first patient in metadata
FIG_ECG  = None   # None = ECG of FIG_ID closest to surgery; or give a file path explicitly
OUT_DIR  = ROOT

INK, MUTED, GRID, BAND = "#0b0b0b", "#52514e", "#e4e3df", "#ecebe7"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8,
                     "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False})

if FIG_ID is None:
    FIG_ID = metadata['ID'].iloc[0]

rows_id = metadata[metadata['ID'] == FIG_ID]
path = FIG_ECG if FIG_ECG else rows_id.loc[rows_id['Duration'].idxmin(), 'ecg_path']

sig = preprocess(path, LEAD, FS)
r_peaks = detect_r_peaks(sig, FS)
residual, n_used, status = qrst_subtract(sig, r_peaks, FS)
fr, fv, m = spectrum(residual, FS)
df_value = fr[m][np.argmax(fv[m])]

to_mV = UV_PER_COUNT / 1000.0
t = np.arange(len(sig)) / FS
print(f"{FIG_ID}: {os.path.basename(path)}, {len(sig)/FS:.0f} s, "
      f"{len(r_peaks)} QRS complexes, {n_used} beats cancelled, DF = {df_value:.2f} Hz")

fig, axes = plt.subplots(3, 1, figsize=(7.0, 6.2), dpi=300,
                         gridspec_kw={"height_ratios": [1, 1, 1.1], "hspace": 0.55})

# A: filtered ECG + R peaks
ax = axes[0]
ax.plot(t, sig * to_mV, color=INK, lw=0.6)
# marker just above the local maximum of each detected complex (±60 ms)
hw = int(0.06 * FS)
gap = 0.08 * (sig.max() - sig.min()) * to_mV
y_mark = np.array([sig[max(0, r-hw):r+hw].max() for r in r_peaks]) * to_mV + gap
ax.plot(r_peaks / FS, y_mark, linestyle='none', marker='v', ms=4,
        color=MUTED, markeredgewidth=0, label='Detected QRS complexes')
ax.set_ylim(None, y_mark.max() + gap)
ax.set_xlim(0, t[-1]); ax.set_ylabel("Amplitude (mV)")
ax.set_xlabel("Time (s)")
ax.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), frameon=False, fontsize=7,
          handletextpad=0.3, borderaxespad=0.1)

# B: residual signal
ax = axes[1]
ax.plot(t, residual * to_mV, color=INK, lw=0.6)
ax.set_xlim(0, t[-1]); ax.set_ylabel("Amplitude (mV)")
ax.set_xlabel("Time (s)")

# C: amplitude spectrum (normalized to the maximum in the search band)
ax = axes[2]
show = fr <= 15
spec = fv / fv[m].max()
ax.axvspan(AF_BAND[0], AF_BAND[1], color=BAND, zorder=0, lw=0)
ax.plot(fr[show], spec[show], color=INK, lw=0.9, zorder=2)
ax.axvline(df_value, color=MUTED, lw=0.9, ls='--', zorder=3)
ax.text(df_value + 0.2, 0.95, f"DF = {df_value:.2f} Hz", color=INK, fontsize=8, va='top')
ax.text((AF_BAND[0] + AF_BAND[1]) / 2, 1.02, "DF search band (3.5–9.0 Hz)",
        color=MUTED, fontsize=7, ha='center', va='bottom', transform=ax.get_xaxis_transform())
ax.set_xlim(0, 15); ax.set_ylim(0, None)
ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("Normalized amplitude")

for ax in axes:
    ax.yaxis.grid(True, color=GRID, lw=0.5); ax.set_axisbelow(True)
for ax, lab in zip(axes, "ABC"):
    ax.text(-0.12, 1.10, lab, transform=ax.transAxes, fontsize=11, fontweight='bold', va='top')

fig.savefig(os.path.join(OUT_DIR, "Figure1_ECG_processing.png"), dpi=1000, bbox_inches='tight')
fig.savefig(os.path.join(OUT_DIR, "Figure1_ECG_processing.tiff"), dpi=1000, bbox_inches='tight',
            pil_kwargs={"compression": "tiff_lzw"})
plt.show()

In [ ]:
# ============================================================
# Supplemental Figure: multiple spectral peaks and peak switching
#   between two same-day ECGs of one patient (lead II)
#   A, B: residual f-wave signals after QRST subtraction
#   C   : amplitude spectra of A and B (+ averaged spectrum of all ECGs of the patient)
# Run after Section 9-5 (uses paths_v, pid_v, dur_v, rate_v, DF_v,
#   preprocess, detect_r_peaks, qrst_subtract, spectrum, Figure 1 style settings).
# ============================================================
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

INK, MUTED, GRID, BAND = "#0b0b0b", "#52514e", "#e4e3df", "#ecebe7"
C1, C2 = "#2a6fdb", "#e0703a"          # recording 1 / recording 2
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8,
                     "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False})
to_mV = UV_PER_COUNT / 1000.0
OUT_DIR = ROOT

def process(path):
    sig = preprocess(path, LEAD, FS)
    res, _, _ = qrst_subtract(sig, detect_r_peaks(sig, FS), FS)
    fr, fv, m = spectrum(res, FS)
    return res, fr, fv, m

def n_peaks(fv, m):
    s = fv[m] / fv[m].max()
    pk, _ = find_peaks(s, prominence=0.1)
    return len(pk)

# ---- Step 1: candidate same-day pairs (both recordings without QRS residue) ----
cand = []
for p in np.unique(pid_v):
    idx = np.where(pid_v == p)[0]
    for a in range(len(idx)):
        for b in range(a + 1, len(idx)):
            i, j = idx[a], idx[b]
            if dur_v[i] == dur_v[j] and rate_v[i] == 0 and rate_v[j] == 0:
                cand.append((i, j, abs(DF_v[i] - DF_v[j])))
cand = sorted(cand, key=lambda x: -x[2])
print(f"Same-day pairs with 0% QRS residue in both: {len(cand)}")

N_SHOW = min(8, len(cand))
fig, axs = plt.subplots(2, 4, figsize=(10, 4.2), dpi=120, sharex=True)
for k, ax in enumerate(axs.flat):
    if k >= N_SHOW:
        ax.axis('off'); continue
    i, j, d = cand[k]
    for ii, col in [(i, C1), (j, C2)]:
        _, fr, fv, m = process(paths_v[ii])
        show = (fr >= 2) & (fr <= 11)
        ax.plot(fr[show], fv[show] / fv[m].max(), color=col, lw=0.8)
        ax.axvline(DF_v[ii], color=col, lw=0.7, ls='--')
    ax.axvspan(*AF_BAND, color=BAND, zorder=0, lw=0)
    ax.set_title(f"#{k}  ΔDF = {d:.2f} Hz", fontsize=8)
fig.supxlabel("Frequency (Hz)"); fig.tight_layout(); plt.show()

# ---- Step 2: figure for the selected pair ----
PICK = 0                                # index (#k) in the candidate panel
i, j, d = cand[PICK]
res1, fr1, fv1, m1 = process(paths_v[i])
res2, fr2, fv2, m2 = process(paths_v[j])

# averaged spectrum of all ECGs of this patient (normalized to band power, as in 9-5)
specs_p = []
for ii in np.where(pid_v == pid_v[i])[0]:
    _, fr, fv, m = process(paths_v[ii])
    specs_p.append(np.interp(fr1, fr, fv) / fv[m].sum())
avg = np.mean(specs_p, axis=0)
n_rec = len(specs_p)
print(f"Selected: {os.path.basename(paths_v[i])} (DF {DF_v[i]:.2f} Hz) / "
      f"{os.path.basename(paths_v[j])} (DF {DF_v[j]:.2f} Hz), ΔDF {d:.2f} Hz, "
      f"ECGs of this patient {n_rec}, number of peaks {n_peaks(fv1, m1)} / {n_peaks(fv2, m2)}")

fig, axes = plt.subplots(3, 1, figsize=(7.0, 6.2), dpi=300,
                         gridspec_kw={"height_ratios": [1, 1, 1.2], "hspace": 0.6})
ylim = 1.1 * max(abs(res1).max(), abs(res2).max()) * to_mV
for ax, res, col, lab in [(axes[0], res1, C1, "Recording 1"), (axes[1], res2, C2, "Recording 2")]:
    t = np.arange(len(res)) / FS
    ax.plot(t, res * to_mV, color=col, lw=0.6)
    ax.set_xlim(0, t[-1]); ax.set_ylim(-ylim, ylim)
    ax.set_ylabel("Amplitude (mV)"); ax.set_xlabel("Time (s)")
    ax.text(1.0, 1.02, lab, transform=ax.transAxes, ha='right', va='bottom', fontsize=7, color=col)

ax = axes[2]
show = fr1 <= 15
ax.axvspan(*AF_BAND, color=BAND, zorder=0, lw=0)
ax.plot(fr1[show], (fv1 / fv1[m1].max())[show], color=C1, lw=0.9, label="Recording 1")
show2 = fr2 <= 15
ax.plot(fr2[show2], (fv2 / fv2[m2].max())[show2], color=C2, lw=0.9, label="Recording 2")
avg_n = avg / avg[m1].max()
ax.plot(fr1[show], avg_n[show], color=INK, lw=0.9, ls=':', label=f"Average of all ECGs (n = {n_rec})")
for val, col in [(DF_v[i], C1), (DF_v[j], C2)]:
    ax.axvline(val, color=col, lw=0.8, ls='--')
ax.text((AF_BAND[0] + AF_BAND[1]) / 2, 1.02, "DF search band (3.5–9.0 Hz)", color=MUTED,
        fontsize=7, ha='center', va='bottom', transform=ax.get_xaxis_transform())
ax.set_xlim(0, 15); ax.set_ylim(0, None)
ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("Normalized amplitude")
ax.legend(loc='upper right', frameon=False, fontsize=7)

for ax in axes:
    ax.yaxis.grid(True, color=GRID, lw=0.5); ax.set_axisbelow(True)
for ax, lab in zip(axes, "ABC"):
    ax.text(-0.12, 1.10, lab, transform=ax.transAxes, fontsize=11, fontweight='bold', va='top')

fig.savefig(os.path.join(OUT_DIR, "SupplementalFigure_multiple_peaks.png"), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(OUT_DIR, "SupplementalFigure_multiple_peaks.tiff"), dpi=300, bbox_inches='tight',
            pil_kwargs={"compression": "tiff_lzw"})
plt.show()


## Appendix: Figure 2 (distribution of DF in lead II)

In [ ]:
# ============================================================
# Figure 2: distribution of DF in lead II by outcome (ECG level)
#   dots: individual ECGs; bar: median; whiskers: interquartile range
# Run after Section 2 (uses DF_v, y_v). Output: 1000 dpi (Heart Rhythm line-art requirement).
# ============================================================
import matplotlib.pyplot as plt

OUT_DIR = ROOT
C_TERM, C_NON = "#2a78d6", "#eb6834"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9,
                     "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False})

groups = [("Termination", 1, C_TERM), ("Non-termination", 0, C_NON)]
fig, ax = plt.subplots(figsize=(3.5, 3.6), dpi=150)
rng = np.random.default_rng(0)
for i, (name, lab, col) in enumerate(groups):
    v = DF_v[y_v == lab]
    x = i + rng.uniform(-0.18, 0.18, len(v))
    ax.scatter(x, v, s=9, color=col, alpha=0.55, linewidths=0, zorder=2)
    q1, med, q3 = np.percentile(v, [25, 50, 75])
    ax.plot([i - 0.28, i + 0.28], [med, med], color=INK, lw=2, zorder=3)
    ax.plot([i, i], [q1, q3], color=INK, lw=1.2, zorder=3)
    for q in (q1, q3):
        ax.plot([i - 0.08, i + 0.08], [q, q], color=INK, lw=1.2, zorder=3)
    print(f"{name}: n = {len(v)} ECGs, median {med:.2f} Hz (IQR {q1:.2f}–{q3:.2f})")
ax.set_xticks([0, 1])
ax.set_xticklabels([f"{n}\n(n = {(y_v == l).sum()} ECGs)" for n, l, _ in groups])
ax.set_xlim(-0.6, 1.6)
ax.set_ylabel("Dominant frequency (Hz)")
ax.set_ylim(3.0, 9.2)
ax.yaxis.grid(True, color=GRID, lw=0.6); ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "Figure2_DF_distribution.png"), dpi=1000)
fig.savefig(os.path.join(OUT_DIR, "Figure2_DF_distribution.tiff"), dpi=1000,
            pil_kwargs={"compression": "tiff_lzw"})
plt.show()
